In [ ]:
from pathlib import Path
import os, sys
import numpy as np
from matplotlib import pyplot as plt
from matplotlib import rcdefaults, gridspec
from matplotlib.ticker import MultipleLocator
import pyvista as pv

# Include this pakage
HaMaGeoLib_DIR = "/home/lochy/ASPECT_PROJECT/HaMaGeoLib"
if os.path.abspath(HaMaGeoLib_DIR) not in sys.path:
    sys.path.append(os.path.abspath(HaMaGeoLib_DIR))
from hamageolib.utils.exception_handler import my_assert
import hamageolib.utils.plot_helper as plot_helper

# Retrieve the default color cycle
default_colors = [color['color'] for color in plt.rcParams['axes.prop_cycle']]

output_dir = "/mnt/lochy/ASPECT_DATA/Collision0/collision_setup_test/test_isostatic_topography_box/img"
topography_files = [
                    # isostacy topography
                    "/mnt/lochy/ASPECT_DATA/Collision0/collision_setup31/C_ar4_SA100.0_CR_CRT2.00e+05_CRV1.00e+20_Ebl1.00e-02_Est1.00e+06_isoI/topography/topography.00001",
                    # "/mnt/lochy/ASPECT_DATA/Collision0/collision_setup30_test/C_ar4_SA100.0_CR_CRT2.00e+05_Ebl1.00e-02_Est1.00e+06_isoI/output/topography/topography.00001"
                    # "/mnt/lochy/ASPECT_DATA/Collision0/collision_setup_test/test_isostatic_topography_box_5/output_particle_L/topography/topography.00000"
                    # "/mnt/lochy/ASPECT_DATA/Collision0/collision_test/fastscape_eroding_box/eroding_box_2d_slope_Ra/topography/topography.00000",
                    # "/mnt/lochy/ASPECT_DATA/Collision0/collision_test/fastscape_eroding_box/eroding_box_2d_slope_Ra/topography/topography.00005",
                    # "/mnt/lochy/ASPECT_DATA/Collision0/collision_test/fastscape_eroding_box/eroding_box_2d_slope_Ra_EBA_1km/topography/topography.00005",
                    # "/mnt/lochy/ASPECT_DATA/Collision0/collision_test/fastscape_eroding_box/eroding_box_slope_2d/topography/topography.00002",
                    # "/mnt/lochy/ASPECT_DATA/Collision0/collision_test/fastscape_eroding_box/eroding_box_2d_slope_Ra/topography/topography.00013",
                    # "/mnt/lochy/ASPECT_DATA/Collision0/collision_test/fastscape_eroding_box/eroding_box_2d_slope_Ra/topography/topography.00023",
                    # "/mnt/lochy/ASPECT_DATA/Collision0/collision_setup_test/test_isostatic_topography_box/output_Mrefined/topography/topography.00000",
                    # "/mnt/lochy/ASPECT_DATA/Collision0/collision_setup_test/test_isostatic_topography_box/output/topography/topography.00000",
                    # "/mnt/lochy/ASPECT_DATA/Collision0/collision_setup_test/test_isostatic_topography_box/output_noI/topography/topography.00003",
                    # Fast scape timestepping, test with WB
                    # "/mnt/lochy/ASPECT_DATA/Collision0/collision_setup_test/MD_lowR/output_main/topography/topography.00012",
                    # "/mnt/lochy/ASPECT_DATA/Collision0/collision_setup_test/FS_lowR/output_main_test/topography/topography.00010",
                    # "/mnt/lochy/ASPECT_DATA/Collision0/collision_setup_test/FS_lowR/output_main_test/topography/topography.00011",
                    # "/mnt/lochy/ASPECT_DATA/Collision0/collision_setup_test/FS_lowR/output_main_test/topography/topography.00012"
                    # Fast scape timestepping, simple test without WB
                    # "/mnt/lochy/ASPECT_DATA/Collision0/collision_setup_test/MD_lowR_1/output_continent_ocean_continent/topography/topography.00010",
                    # "/mnt/lochy/ASPECT_DATA/Collision0/collision_setup_test/FS_lowR_1/output_continent_ocean_continent/topography/topography.00009",
                    # "/mnt/lochy/ASPECT_DATA/Collision0/collision_setup_test/FS_lowR_1/output_continent_ocean_continent/topography/topography.00010"
                    ]

# Example usage
# Rule of thumbs:
# 1. Set the limit to something like 5.0, 10.0 or 50.0, 100.0 
# 2. Set five major ticks for each axis
scaling_factor = 1.0  # scale factor of plot
font_scaling_multiplier = 1.5 # extra scaling multiplier for font
legend_font_scaling_multiplier = 0.5
line_width_scaling_multiplier = 2.0 # extra scaling multiplier for lines
n_minor_ticks = 4  # number of minor ticks between two major ones

# scale the matplotlib params
plot_helper.scale_matplotlib_params(scaling_factor, font_scaling_multiplier=font_scaling_multiplier,\
                        legend_font_scaling_multiplier=legend_font_scaling_multiplier,
                        line_width_scaling_multiplier=line_width_scaling_multiplier)

# Update font settings for compatibility with publishing tools like Illustrator.
plt.rcParams.update({
    'font.family': 'Times New Roman',
    'pdf.fonttype': 42,
    'ps.fonttype': 42
})

# create the plot
fig, ax = plt.subplots(figsize=(10, 6))
# ax_twinx = ax.twinx()

# options of plot
x_lim = [0, 5500]
# x_lim = [0, 5500]; x_half_width = None  # km, options for plotting the whole width
# x_lim = None; x_half_width = 500 # km, options for plotting around the suture
topo_lim = [-5000, 5000]  # m
topo_tick_interval = 1000.0
deformation_ratio_lim = [-0.5, 0.5]
deformation_ratio_tick_interval = 0.1


for i, topography_file in enumerate(topography_files):

    # Extract topography data
    my_assert(os.path.isfile(topography_file), FileExistsError, "%s doesn't exist." % topography_file)

    print("Read from %s" % topography_file)
    data = np.loadtxt(topography_file, comments="#")

    x = data[:, 0]
    topography = data[:, 2] 
    
    # creat the plot
    ax.plot(x/1e3, topography, label="topography %d" % i, color=default_colors[i])


# adjust plot axis
ax.set_xlabel("X (km)")
ax.set_xlim(x_lim)

ax.set_ylabel("Topography (m)") # , color=default_colors[0])
ax.set_ylim(topo_lim)
ax.yaxis.set_major_locator(MultipleLocator(topo_tick_interval))
ax.yaxis.set_minor_locator(MultipleLocator(topo_tick_interval/(n_minor_ticks+1)))

ax.legend()

ax.grid()

fig_path = os.path.join(output_dir, "topography")
fig.savefig(fig_path + ".png")
print("Saved figure %s" % (fig_path + ".png"))

fig.savefig(fig_path + ".pdf")
print("Saved figure %s" % (fig_path + ".pdf"))

rcdefaults()

In [ ]:
import pyvista as pv
import numpy as np

# pvtu_path = "/mnt/lochy/ASPECT_DATA/Collision0/collision_setup_test/FS_lowR_1/output_continent_ocean_continent/solution/solution-00008.pvtu"
pvtu_path = "/mnt/lochy/ASPECT_DATA/Collision0/collision_setup_test/FS_lowR/output_main_test/solution/solution-00008.pvtu"
refinement = 2+1 # global + adaptive

mesh = pv.read(pvtu_path)

points = mesh.points
x = points[:, 0]
y = points[:, 1]

# Highest y coordinate
ymax = np.max(y)
resolution = ymax / 2**refinement / 2.0  # 2-order mesh

# Numerical tolerance
tol = 10e3

# Top points at model surface
top_mask = np.abs(y - ymax) < tol
top_points = points[top_mask]
topo = top_points[:, 1]-2000e3
x_surf = top_points[:, 0]
dtopo_dx = np.gradient(topo, x_surf)

# Extract velocities at the top points
velocity = mesh["velocity"][top_mask]
vx = velocity[:, 0]
vy = velocity[:, 1]

print(f"Found {top_points.shape[0]} top boundary points.")

In [ ]:

# Example usage
# Rule of thumbs:
# 1. Set the limit to something like 5.0, 10.0 or 50.0, 100.0 
# 2. Set five major ticks for each axis
scaling_factor = 1.0  # scale factor of plot
font_scaling_multiplier = 1.5 # extra scaling multiplier for font
legend_font_scaling_multiplier = 0.5
line_width_scaling_multiplier = 2.0 # extra scaling multiplier for lines
n_minor_ticks = 4  # number of minor ticks between two major ones

# scale the matplotlib params
plot_helper.scale_matplotlib_params(scaling_factor, font_scaling_multiplier=font_scaling_multiplier,\
                        legend_font_scaling_multiplier=legend_font_scaling_multiplier,
                        line_width_scaling_multiplier=line_width_scaling_multiplier)

# Update font settings for compatibility with publishing tools like Illustrator.
plt.rcParams.update({
    'font.family': 'Times New Roman',
    'pdf.fonttype': 42,
    'ps.fonttype': 42
})

fig = plt.figure(figsize=(10, 12))
gs = gridspec.GridSpec(2, 1)

# subfigure - plot the topography with the uplift rates
ax = fig.add_subplot(gs[0, 0])
ax_twinx = ax.twinx() # velocity

ax.scatter(top_points[:, 0]/1e3, (topo), s=5, color=default_colors[0], label="Topo")
ax_twinx.scatter(top_points[:, 0]/1e3, vy, s=5, color=default_colors[1], label="Vy")

ax.set_xlabel("X (km)")
ax.set_xlim(x_lim)

ax.set_ylabel("Topo (m)")
ax.set_ylim(topo_lim)
ax.yaxis.set_major_locator(MultipleLocator(topo_tick_interval))
ax.yaxis.set_minor_locator(MultipleLocator(topo_tick_interval/(n_minor_ticks+1)))

vy_lim = [-0.1, 0.1]
vy_tick_interval = 0.02
ax_twinx.set_ylabel("Vy (m/yr)")
ax_twinx.set_ylim(vy_lim)
ax_twinx.yaxis.set_major_locator(MultipleLocator(vy_tick_interval))
ax_twinx.yaxis.set_minor_locator(MultipleLocator(vy_tick_interval/(n_minor_ticks+1)))

handles1, labels1 = ax.get_legend_handles_labels()
handles2, labels2 = ax_twinx.get_legend_handles_labels()

ax.legend(handles1 + handles2, labels1 + labels2, loc="best")
ax.grid()

# subfigure - plot the topography gradient
ax1 = fig.add_subplot(gs[1, 0])

ax1.scatter(x_surf/1e3, (dtopo_dx), s=5, color=default_colors[0], label="d_Topo_dx")

ax1.set_xlabel("X (km)")
ax1.set_xlim(x_lim)

ax1.set_ylabel("Topo Gradient")
# ax1.set_ylim(topo_lim)
# ax1.yaxis.set_major_locator(MultipleLocator(topo_tick_interval))
# ax1.yaxis.set_minor_locator(MultipleLocator(topo_tick_interval/(n_minor_ticks+1)))

ax1.legend()
ax1.grid()

fig_path = os.path.join(os.path.dirname(pvtu_path), 
                        "surface_velocity_%s.png" % (os.path.basename(pvtu_path)).split(".")[0])
fig.savefig(fig_path)
print("Saved %s" % fig_path)

rcdefaults()

relief = np.max(topo) - np.min(topo)
print(f"relief = {relief}m")

vx_max = np.max(vx)
print(f"vy_max = {vx_max}m/yr")

vy_max = np.max(vy)
print(f"vy_max = {vy_max}m/yr")

# note advection is only along x
# h_time has parallel meating to timestepping
# while v_time is just something I want to checkt
h_time = resolution / vx_max 
print(f"h_time (by resolution) = {h_time}yr")

v_time = relief / vy_max
print(f"v_time (by relief) = {v_time}yr")

In [ ]:
# This is the biggest differences in topography between adjacent points
dtopo_dx_diff_max = 0.008
current_timestep = 1.139062500000e+03 # yr

dtopo_dx_diff_max_dx = dtopo_dx_diff_max / resolution

# (vx_max * current_timestep * dtopo_dx_diff_max) - a discrepency in height
diff = (vx_max * current_timestep) * dtopo_dx_diff_max
diff1 = (vx_max * current_timestep)**2.0 * dtopo_dx_diff_max_dx

print(f"diff={diff}m")
print(f"diff1={diff1}m")

In [ ]:
print(resolution)
print(vx_max * current_timestep)